# Metodología  
1. hacer reducción dimensional con regularización lasso o con Randon Forest
2. 

# Secuencia metodológica seguida en el algoritmo anterior  

# Secuencia Metodológica de Machine Learning - LightGBM

Este script implementa una metodología completa de machine learning para predecir casos de dengue, con un enfoque especial en la detección de picos epidémicos. A continuación, se detalla la secuencia metodológica paso a paso:

---



## **FASE 1: PREPARACIÓN Y CONFIGURACIÓN INICIAL**

### 1.1 Configuración del Entorno
- **Importación de librerías**: Se cargan todas las bibliotecas necesarias para el pipeline completo (pandas, numpy, scikit-learn, lightgbm, optuna, etc.)
- **Definición de rutas**: Se establecen directorios para datos de entrada, procesados y resultados
- **Configuración de warnings**: Se suprimen advertencias para una salida más limpia

### 1.2 Carga y Exploración de Datos
- **Carga de datos**: Lectura del archivo Excel con datos históricos (2021-2026)
- **Identificación de variables**:
  - Variable objetivo: `casos_dengue`
  - Variables excluidas: `fecha`, `año`, `semana_epi`
  - Variables predictoras: Todas las demás columnas (variables meteorológicas y epidemiológicas)

---



## **FASE 2: INGENIERÍA DE ATRIBUTOS AVANZADA**

### 2.1 Características de Tendencia
- Media móvil de 13 semanas (`tendencia`)
- Lags de la tendencia (1, 2, 3 semanas)

### 2.2 Características de Estacionalidad
- Lags estacionales (4, 8, 12, 16, 20, 24 semanas)
- Diferencias estacionales (valor actual - valor en lag específico)

### 2.3 Características de Cambio y Aceleración
- Diferencias de primer, segundo y tercer orden
- Aceleración (segunda diferencia)

### 2.4 Características de Ratio y Cambio Porcentual
- Cambios porcentuales (1 y 2 semanas)
- Ratios con lag (valor actual / valor anterior + 1)

### 2.5 Características de Ventanas Móviles
- Media, desviación estándar, máximo y mínimo para ventanas de: 3, 5, 7, 13, 26 semanas

### 2.6 Características de Rango
- Rango (max - min) para ventanas de 7 y 13 semanas

### 2.7 Características de Picos (Detección de Anomalías)
- Cálculo de Z-scores para identificar valores atípicos
- Variable binaria `es_pico` (Z-score > 2)
- Lags de la variable pico (1 y 2 semanas)

### 2.8 Características de Interacción
- Producto de variables meteorológicas con casos en lag 1
- Producto de variables meteorológicas con tendencia

---



## **FASE 3: SELECCIÓN DE CARACTERÍSTICAS**

### 3.1 Selección con Random Forest
- **Entrenamiento de RF**: Modelo con 100 árboles para evaluar importancia de características
- **Cálculo de importancia**: Uso de `feature_importances_` para rankear predictores
- **Selección de top features**: Se mantienen las 30 características más importantes (máximo)
- **Justificación**: Reducción de dimensionalidad, mejora de interpretabilidad y eficiencia computacional

### 3.2 Almacenamiento de Resultados
- Guardado de la lista de características seleccionadas en Excel
- Documentación del proceso de selección

---



## **FASE 4: PREPARACIÓN DE DATOS FINALES**

### 4.1 División Train/Test
- **Entrenamiento**: Datos 2021-2025
- **Test**: Datos 2026
- **Justificación**: Evaluación temporal realista (predicción fuera de muestra)

### 4.2 Configuración de Validación para Optimización
- **Validación**: Datos 2021-2024
- **Optimización**: Datos 2025
- **Estrategia**: Validación temporal separada para evitar data leakage

---



## **FASE 5: OPTIMIZACIÓN DE HIPERPARÁMETROS**

### 5.1 Función de Pesos Avanzada
**Objetivo**: Dar mayor importancia a casos críticos durante el entrenamiento

**Estrategias de ponderación**:
1. **Pesos por magnitud**:
   - Percentil 80%: Peso 1.5-3.0 (lineal)
   - Percentil 95%: Peso 3.0-5.0 (exponencial)

2. **Pesos por tendencia ascendente**:
   - Incremento del 30% para semanas con tendencia creciente de 3 semanas

3. **Pesos por incrementos significativos**:
   - Incremento del 20% para diferencias > percentil 80%

### 5.2 Optimización con Optuna
**Algoritmo**: TPE (Tree-structured Parzen Estimator)

**Hiperparámetros optimizados**:
- `num_leaves`: 20-80 (complejidad del árbol)
- `learning_rate`: 0.005-0.05 (tasa de aprendizaje logarítmica)
- `feature_fraction`: 0.6-1.0 (submuestreo de características)
- `bagging_fraction`: 0.6-1.0 (submuestreo de datos)
- `bagging_freq`: 1-10 (frecuencia de bagging)
- `min_child_samples`: 5-30 (muestras mínimas por hoja)
- `reg_alpha`, `reg_lambda`: 0-2 (regularización L1/L2)
- `min_split_gain`: 0-0.5 (ganancia mínima para división)
- `max_depth`: 5-12 (profundidad máxima)
- `subsample`, `colsample_bytree`: 0.6-1.0 (muestreo adicional)

### 5.3 Función Objetivo Combinada
**Métrica compuesta**:
- 70% MAE general
- 30% MAE en picos (> percentil 80)
- 10% Penalización por subestimación severa en picos

**Número de trials**: 50 iteraciones de optimización

---



## **FASE 6: ENTRENAMIENTO DEL MODELO FINAL**

### 6.1 Configuración del Modelo
- **Parámetros base**: Objective='regression', metric='mae', boosting='gbdt'
- **Parámetros optimizados**: Incorporación de mejores hiperparámetros
- **Early stopping**: 200 rounds sin mejora
- **Número de rounds**: 5000 (con parada temprana)

### 6.2 Entrenamiento con Pesos
- Aplicación de pesos avanzados al conjunto de entrenamiento
- Validación en conjunto de test (2026)

### 6.3 Predicciones
- Predicción en entrenamiento (2021-2025)
- Predicción en test (2026)

---



## **FASE 7: EVALUACIÓN DEL MODELO**

### 7.1 Métricas de Rendimiento
- **MAE** (Error Absoluto Medio): General y en picos
- **RMSE** (Raíz del Error Cuadrático Medio)
- **R²** (Coeficiente de Determinación)

### 7.2 Métricas Específicas para Picos
- Cálculo de MAE para valores > percentil 80%
- Evaluación de capacidad de detección de brotes

### 7.3 Análisis de Errores por Rango
- Segmentación por niveles de casos: 0-5, 5-10, 10-20, 20-50, 50-100, 100-200
- Cálculo de MAE y error máximo por segmento
- Identificación de patrones de error

---


# Versión corregida una vez 

In [6]:
from pathlib import Path

# LIGHTGBM PARA PREDICCIÓN DE CASOS DE DENGUE
# VERSIÓN CORREGIDA: VALIDACIÓN TEMPORAL + CONTROL DE DATA LEAKAGE
# ================================================================
#
# Principales correcciones:
# 1. Los atributos derivados de casos de dengue usan SOLO información pasada.
# 2. No se utiliza casos_dengue_t para construir atributos de y_t.
# 3. La selección de variables con Random Forest se hace SOLO con TRAIN.
# 4. El detector de picos se calibra SOLO con los datos disponibles en TRAIN.
# 5. La optimización usa validación walk-forward.
# 6. El conjunto 2026 se reserva exclusivamente para evaluación final.
# 7. La importancia de RF se guarda correctamente.
# 8. Se incluyen baselines para saber si LightGBM realmente aporta valor.
# 9. Se calculan MAE, RMSE, R², Peak MAE, sesgo y métricas por rango.
#
# NOTA:
# Este script supone que el archivo contiene:
#   fecha, año, semana_epi, casos_dengue
# y variables meteorológicas.
# ================================================================

import os
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

# ================================================================
# 1. CONFIGURACIÓN
# ================================================================

input_file = (
    r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina"
    r"\4_algoritmo_LightGBM\2_datos"
    r"\1_raw\2_meteo_epi_2021-2026_1_rezagos.xlsx"
)

output_dir = (
    r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina"
    r"\4_algoritmo_LightGBM\3_resultados"
)

processed_dir = (
    r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina"
    r"\4_algoritmo_LightGBM\2_datos\2_procesados"
)

os.makedirs(output_dir, exist_ok=True)
os.makedirs(processed_dir, exist_ok=True)

RANDOM_STATE = 42
TEST_YEAR = 2026

# Número máximo de variables seleccionadas
MAX_SELECTED_FEATURES = 30

# Número de ensayos Optuna
N_TRIALS = 50

# ================================================================
# 2. CARGA DE DATOS
# ================================================================

print("=" * 80)
print("CARGANDO DATOS")
print("=" * 80)

df = pd.read_excel(input_file)

if "fecha" not in df.columns:
    raise ValueError("No existe la columna 'fecha'.")

if "casos_dengue" not in df.columns:
    raise ValueError("No existe la columna 'casos_dengue'.")

df["fecha"] = pd.to_datetime(df["fecha"])

# Orden cronológico obligatorio
df = df.sort_values("fecha").reset_index(drop=True)

target_col = "casos_dengue"
exclude_cols = ["fecha", "año", "semana_epi"]

# Si año no existe, derivarlo de fecha
if "año" not in df.columns:
    df["año"] = df["fecha"].dt.year

# Si semana_epi no existe, construir una semana ISO
if "semana_epi" not in df.columns:
    df["semana_epi"] = df["fecha"].dt.isocalendar().week.astype(int)

predictor_cols = [
    col for col in df.columns
    if col not in exclude_cols + [target_col]
]

print(f"Registros originales: {len(df)}")
print(f"Predictores meteorológicos/iniciales: {len(predictor_cols)}")
print(f"Periodo: {df['fecha'].min()} → {df['fecha'].max()}")

# ================================================================
# 3. INGENIERÍA DE ATRIBUTOS SIN DATA LEAKAGE
# ================================================================
#
# Regla fundamental:
#
# Para predecir y_t:
#
#     X_t = información disponible hasta t-1
#
# Por tanto, cualquier atributo construido a partir de
# casos_dengue debe utilizar shift(1) o rezagos posteriores.
# ================================================================

print("\n" + "=" * 80)
print("INGENIERÍA DE ATRIBUTOS SIN DATA LEAKAGE")
print("=" * 80)

df_engineered = df.copy()

y = df_engineered[target_col]

# ------------------------------------------------
# 3.1 Rezagos del target
# ------------------------------------------------

for lag in [1, 2, 3, 4, 8, 12, 16, 20, 24, 26, 52]:
    df_engineered[f"casos_lag_{lag}"] = y.shift(lag)

# ------------------------------------------------
# 3.2 Tendencia
# ------------------------------------------------
#
# CORRECCIÓN:
# antes se calculaba rolling incluyendo y_t.
# Ahora se desplaza primero.
# ------------------------------------------------

for window in [3, 5, 7, 13, 26]:
    df_engineered[f"roll_mean_{window}"] = (
        y.shift(1).rolling(window=window, min_periods=window).mean()
    )

    df_engineered[f"roll_std_{window}"] = (
        y.shift(1).rolling(window=window, min_periods=window).std()
    )

    df_engineered[f"roll_max_{window}"] = (
        y.shift(1).rolling(window=window, min_periods=window).max()
    )

    df_engineered[f"roll_min_{window}"] = (
        y.shift(1).rolling(window=window, min_periods=window).min()
    )

# Tendencia de 13 semanas
df_engineered["tendencia"] = (
    y.shift(1).rolling(window=13, min_periods=13).mean()
)

for lag in [1, 2, 3]:
    df_engineered[f"tendencia_lag{lag}"] = (
        df_engineered["tendencia"].shift(lag)
    )

# ------------------------------------------------
# 3.3 Diferencias temporales SIN usar y_t
# ------------------------------------------------

for lag in [1, 2, 3, 4, 8, 12, 24]:
    df_engineered[f"diff_lag_{lag}"] = (
        y.shift(1) - y.shift(lag + 1)
    )

# Cambios semana a semana disponibles antes de t
df_engineered["diff_1"] = y.shift(1) - y.shift(2)
df_engineered["diff_2"] = y.shift(2) - y.shift(3)
df_engineered["diff_3"] = y.shift(3) - y.shift(4)

# Aceleración basada exclusivamente en pasado
df_engineered["acceleration"] = (
    df_engineered["diff_1"] - df_engineered["diff_2"]
)

# ------------------------------------------------
# 3.4 Cambios porcentuales SIN usar y_t
# ------------------------------------------------

for lag in [1, 2, 4, 12]:
    denominator = y.shift(lag + 1).replace(0, np.nan)

    df_engineered[f"pct_change_lag_{lag}"] = (
        (y.shift(1) - y.shift(lag + 1)) /
        denominator
    ) * 100

# Ratio entre observaciones pasadas
df_engineered["ratio_lag1"] = (
    y.shift(1) / (y.shift(2) + 1)
)

# ------------------------------------------------
# 3.5 Rangos históricos
# ------------------------------------------------

df_engineered["range_7"] = (
    df_engineered["roll_max_7"] -
    df_engineered["roll_min_7"]
)

df_engineered["range_13"] = (
    df_engineered["roll_max_13"] -
    df_engineered["roll_min_13"]
)

# ------------------------------------------------
# 3.6 Indicadores de pico
# ------------------------------------------------
#
# IMPORTANTE:
# NO calculamos el z-score usando 2021-2026 completo.
# El umbral se estimará posteriormente usando SOLO TRAIN.
# Aquí únicamente dejamos los rezagos del indicador.
#
# El indicador se construirá después de la separación temporal.
# ------------------------------------------------

# ------------------------------------------------
# 3.7 Interacciones meteorología × información pasada
# ------------------------------------------------

meteo_vars = [
    col for col in predictor_cols
    if col.startswith(
        ("prec", "temp", "tmax", "tmin", "hr", "soi", "oni", "mei")
    )
]

# Limitar las interacciones para controlar dimensionalidad.
# Se seleccionan hasta 5 variables disponibles.
for var in meteo_vars[:5]:
    df_engineered[f"{var}_x_casos_lag1"] = (
        df_engineered[var] * y.shift(1)
    )

    df_engineered[f"{var}_x_tendencia"] = (
        df_engineered[var] *
        df_engineered["tendencia"]
    )

# ------------------------------------------------
# 3.8 Variables de calendario
# ------------------------------------------------

if "semana_epi" in df_engineered.columns:
    # Representación cíclica de la semana epidemiológica
    df_engineered["semana_sin"] = np.sin(
        2 * np.pi * df_engineered["semana_epi"] / 52
    )

    df_engineered["semana_cos"] = np.cos(
        2 * np.pi * df_engineered["semana_epi"] / 52
    )

# ================================================================
# 4. SEPARACIÓN TEMPORAL
# ================================================================
#
# 2021-2025 = TRAIN
# 2026      = TEST FINAL
#
# El test de 2026 no se toca durante selección de variables,
# optimización ni calibración de umbrales.
# ================================================================

df_engineered = df_engineered.replace([np.inf, -np.inf], np.nan)

train_mask = (
    (df_engineered["año"] >= 2021) &
    (df_engineered["año"] <= 2025)
)

test_mask = df_engineered["año"] == TEST_YEAR

print("\n" + "=" * 80)
print("SEPARACIÓN TEMPORAL")
print("=" * 80)

print(f"TRAIN: {train_mask.sum()} registros")
print(f"TEST : {test_mask.sum()} registros")

if test_mask.sum() == 0:
    raise ValueError("No se encontraron registros para el año 2026.")

# ================================================================
# 5. CONSTRUCCIÓN DEL INDICADOR DE PICO SIN LEAKAGE
# ================================================================
#
# El umbral se calcula exclusivamente con TRAIN.
# Posteriormente se aplica tanto a train como a test.
# ================================================================

train_y_raw = df_engineered.loc[train_mask, target_col]

peak_threshold = np.percentile(train_y_raw, 80)
extreme_threshold = np.percentile(train_y_raw, 95)

print(f"Umbral pico (80% TRAIN): {peak_threshold:.3f}")
print(f"Umbral extremo (95% TRAIN): {extreme_threshold:.3f}")

df_engineered["es_pico_base"] = (
    y >= peak_threshold
).astype(int)

# Para predecir t utilizamos solamente indicadores de semanas anteriores
df_engineered["es_pico_lag1"] = (
    df_engineered["es_pico_base"].shift(1)
)

df_engineered["es_pico_lag2"] = (
    df_engineered["es_pico_base"].shift(2)
)

# El indicador actual no debe entrar como predictor.
df_engineered.drop(columns=["es_pico_base"], inplace=True)

# ================================================================
# 6. ELIMINAR NULOS
# ================================================================

df_engineered = df_engineered.dropna().reset_index(drop=True)

# Recalcular máscaras después de dropna
train_mask = (
    (df_engineered["año"] >= 2021) &
    (df_engineered["año"] <= 2025)
)

test_mask = df_engineered["año"] == TEST_YEAR

new_predictor_cols = [
    col for col in df_engineered.columns
    if col not in exclude_cols + [target_col]
]

print("\n" + "=" * 80)
print("DATOS DESPUÉS DE INGENIERÍA")
print("=" * 80)

print(f"Predictores: {len(new_predictor_cols)}")
print(f"TRAIN: {train_mask.sum()}")
print(f"TEST : {test_mask.sum()}")

# ================================================================
# 7. MATRICES DE DATOS
# ================================================================

X_all = df_engineered[new_predictor_cols]
y_all = df_engineered[target_col]

X_train_full = X_all.loc[train_mask].copy()
y_train_full = y_all.loc[train_mask].copy()

X_test = X_all.loc[test_mask].copy()
y_test = y_all.loc[test_mask].copy()

fechas_train = df_engineered.loc[train_mask, "fecha"].values
fechas_test = df_engineered.loc[test_mask, "fecha"].values

años_train = df_engineered.loc[train_mask, "año"].values
años_test = df_engineered.loc[test_mask, "año"].values

semanas_train = df_engineered.loc[train_mask, "semana_epi"].values
semanas_test = df_engineered.loc[test_mask, "semana_epi"].values

# ================================================================
# 8. SELECCIÓN DE FEATURES CON RANDOM FOREST
# ================================================================
#
# CORRECCIÓN CRÍTICA:
# RF solamente ve TRAIN.
# Nunca observa 2026.
# ================================================================

print("\n" + "=" * 80)
print("SELECCIÓN DE FEATURES CON RANDOM FOREST — SOLO TRAIN")
print("=" * 80)

rf = RandomForestRegressor(
    n_estimators=500,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    max_features="sqrt"
)

rf.fit(X_train_full, y_train_full)

feature_importance = pd.DataFrame({
    "Feature": X_train_full.columns,
    "Importance": rf.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

n_selected = min(
    MAX_SELECTED_FEATURES,
    len(feature_importance)
)

selected_features_rf = (
    feature_importance
    .head(n_selected)["Feature"]
    .tolist()
)

print(f"Features iniciales: {len(new_predictor_cols)}")
print(f"Features seleccionadas: {len(selected_features_rf)}")

print("\nTop 15:")
for i, row in feature_importance.head(15).iterrows():
    print(
        f"{i+1:2d}. "
        f"{row['Feature']:<40} "
        f"{row['Importance']:.6f}"
    )

# Guardar importancia correcta
features_file = os.path.join(
    output_dir,
    "atributos_seleccionados_rf_sin_leakage.xlsx"
)

feature_importance.to_excel(
    features_file,
    index=False
)

# Reducir matrices
X_train_full = X_train_full[selected_features_rf]
X_test = X_test[selected_features_rf]

# ================================================================
# 9. FUNCIÓN DE PESOS
# ================================================================

def create_advanced_weights(
    y,
    peak_threshold,
    extreme_threshold,
    increase_weight=1.3
):
    """
    Construye pesos a partir exclusivamente de y del conjunto
    que se está utilizando para entrenar.

    Los umbrales se calibran externamente con TRAIN.
    """

    y = np.asarray(y)

    weights = np.ones(len(y), dtype=float)

    # Valores altos
    for i, val in enumerate(y):

        if val >= extreme_threshold:
            denominator = max(extreme_threshold + 1, 1)

            weight_factor = (
                3.0 +
                ((val - extreme_threshold) / denominator) * 2.0
            )

            weights[i] *= weight_factor

        elif val >= peak_threshold:

            denominator = max(peak_threshold + 1, 1)

            weight_factor = (
                1.5 +
                ((val - peak_threshold) / denominator) * 1.5
            )

            weights[i] *= weight_factor

    # Tendencia ascendente fuerte
    if len(y) > 3:
        for i in range(3, len(y)):
            if (
                y[i] > y[i-1]
                and y[i] > y[i-2]
                and y[i] > y[i-3]
            ):
                weights[i] *= increase_weight

    # Incrementos importantes
    if len(y) > 1:
        diff = np.diff(y)

        diff_high = np.percentile(
            np.abs(diff),
            80
        )

        for i in range(1, len(y)):
            if diff[i-1] > diff_high:
                weights[i] *= 1.2

    return weights


# ================================================================
# 10. VALIDACIÓN WALK-FORWARD
# ================================================================
#
# Usamos:
#
# Fold 1:
#   Train 2021-2022 → Validación 2023
#
# Fold 2:
#   Train 2021-2023 → Validación 2024
#
# Fold 3:
#   Train 2021-2024 → Validación 2025
#
# El año 2026 queda completamente aislado.
# ================================================================

def get_walk_forward_folds(df_data):
    folds = []

    years = sorted(
        df_data.loc[train_mask, "año"].unique()
    )

    for val_year in years[2:]:
        train_years = [
            year for year in years
            if year < val_year
        ]

        if len(train_years) < 2:
            continue

        train_idx = df_data.index[
            df_data["año"].isin(train_years)
        ].to_numpy()

        val_idx = df_data.index[
            df_data["año"] == val_year
        ].to_numpy()

        if len(train_idx) > 0 and len(val_idx) > 0:
            folds.append(
                (
                    train_idx,
                    val_idx,
                    train_years,
                    val_year
                )
            )

    return folds


folds = get_walk_forward_folds(df_engineered)

print("\n" + "=" * 80)
print("FOLDS WALK-FORWARD")
print("=" * 80)

for i, (_, _, train_years, val_year) in enumerate(folds, 1):
    print(
        f"Fold {i}: "
        f"Train {train_years} → "
        f"Validación {val_year}"
    )

# ================================================================
# 11. OPTUNA
# ================================================================

print("\n" + "=" * 80)
print("OPTIMIZACIÓN LIGHTGBM CON OPTUNA")
print("=" * 80)


def objective(trial):

    params = {
        "objective": "regression",
        "metric": "mae",
        "boosting_type": "gbdt",

        "num_leaves": trial.suggest_int(
            "num_leaves", 10, 80
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.005,
            0.05,
            log=True
        ),

        "feature_fraction": trial.suggest_float(
            "feature_fraction",
            0.6,
            1.0
        ),

        "bagging_fraction": trial.suggest_float(
            "bagging_fraction",
            0.6,
            1.0
        ),

        "bagging_freq": trial.suggest_int(
            "bagging_freq",
            1,
            10
        ),

        "min_child_samples": trial.suggest_int(
            "min_child_samples",
            5,
            40
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            0.0,
            2.0
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            0.0,
            2.0
        ),

        "min_split_gain": trial.suggest_float(
            "min_split_gain",
            0.0,
            0.5
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            12
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.6,
            1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0
        ),

        "verbosity": -1,
        "n_jobs": -1,
        "random_state": RANDOM_STATE
    }

    fold_scores = []

    for fold_number, (
        train_idx,
        val_idx,
        train_years,
        val_year
    ) in enumerate(folds, 1):

        # Convert indices from original dataframe to positions
        train_positions = df_engineered.index.get_indexer(
            train_idx
        )

        val_positions = df_engineered.index.get_indexer(
            val_idx
        )

        X_tr = X_all.loc[train_idx, selected_features_rf]
        y_tr = y_all.loc[train_idx]

        X_val = X_all.loc[val_idx, selected_features_rf]
        y_val = y_all.loc[val_idx]

        # Pesos SOLO para observaciones de entrenamiento
        # usando umbrales calibrados en TRAIN.
        weights_tr = create_advanced_weights(
            y_tr.values,
            peak_threshold,
            extreme_threshold
        )

        dtrain = lgb.Dataset(
            X_tr,
            label=y_tr,
            weight=weights_tr
        )

        dval = lgb.Dataset(
            X_val,
            label=y_val,
            reference=dtrain
        )

        model = lgb.train(
            params,
            dtrain,
            num_boost_round=3000,
            valid_sets=[dval],
            valid_names=["validation"],
            callbacks=[
                lgb.early_stopping(
                    100,
                    verbose=False
                ),
                lgb.log_evaluation(0)
            ]
        )

        pred = model.predict(
            X_val,
            num_iteration=model.best_iteration
        )

        general_mae = mean_absolute_error(
            y_val,
            pred
        )

        peak_mask = (
            y_val.values >= peak_threshold
        )

        if peak_mask.sum() > 0:
            peak_mae = mean_absolute_error(
                y_val.values[peak_mask],
                pred[peak_mask]
            )
        else:
            peak_mae = general_mae

        severe_under = (
            np.mean(
                np.maximum(
                    0,
                    y_val.values[peak_mask] -
                    pred[peak_mask]
                )
            )
            if peak_mask.sum() > 0
            else 0.0
        )

        combined_score = (
            0.70 * general_mae +
            0.30 * peak_mae +
            0.10 * severe_under
        )

        fold_scores.append(combined_score)

    return float(np.mean(fold_scores))


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(
        seed=RANDOM_STATE
    )
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True
)

best_params = study.best_params

print("\nMejores parámetros:")
for key, value in best_params.items():
    print(f"  {key}: {value}")

print(f"\nMejor score walk-forward: {study.best_value:.4f}")

# ================================================================
# 12. ENTRENAMIENTO FINAL
# ================================================================
#
# Después de terminar toda la optimización:
#
# 2021-2025 → entrenamiento final
# 2026      → test
#
# 2026 NO participa en early stopping.
#
# Para evitar utilizar el test durante entrenamiento, usamos un
# número de iteraciones estimado a partir de la mediana de las
# mejores iteraciones obtenidas durante la validación walk-forward.
# ================================================================

print("\n" + "=" * 80)
print("ENTRENAMIENTO FINAL")
print("=" * 80)

best_iterations = []

for fold_number, (
    train_idx,
    val_idx,
    train_years,
    val_year
) in enumerate(folds, 1):

    X_tr = X_all.loc[
        train_idx,
        selected_features_rf
    ]

    y_tr = y_all.loc[train_idx]

    X_val = X_all.loc[
        val_idx,
        selected_features_rf
    ]

    y_val = y_all.loc[val_idx]

    weights_tr = create_advanced_weights(
        y_tr.values,
        peak_threshold,
        extreme_threshold
    )

    dtrain = lgb.Dataset(
        X_tr,
        label=y_tr,
        weight=weights_tr
    )

    dval = lgb.Dataset(
        X_val,
        label=y_val,
        reference=dtrain
    )

    fold_params = {
        "objective": "regression",
        "metric": "mae",
        "boosting_type": "gbdt",
        "verbosity": -1,
        "n_jobs": -1,
        "random_state": RANDOM_STATE,
        **best_params
    }

    fold_model = lgb.train(
        fold_params,
        dtrain,
        num_boost_round=3000,
        valid_sets=[dval],
        valid_names=["validation"],
        callbacks=[
            lgb.early_stopping(
                100,
                verbose=False
            ),
            lgb.log_evaluation(0)
        ]
    )

    best_iterations.append(
        fold_model.best_iteration
    )

if len(best_iterations) > 0:
    final_num_boost_round = int(
        np.median(best_iterations)
    )
else:
    final_num_boost_round = 500

# Evitar valores extremos
final_num_boost_round = max(
    50,
    min(final_num_boost_round, 3000)
)

print(
    f"Mejores iteraciones por fold: "
    f"{best_iterations}"
)

print(
    f"Número de iteraciones finales: "
    f"{final_num_boost_round}"
)

final_params = {
    "objective": "regression",
    "metric": "mae",
    "boosting_type": "gbdt",
    "verbosity": -1,
    "n_jobs": -1,
    "random_state": RANDOM_STATE,
    **best_params
}

train_weights = create_advanced_weights(
    y_train_full.values,
    peak_threshold,
    extreme_threshold
)

dtrain_final = lgb.Dataset(
    X_train_full,
    label=y_train_full,
    weight=train_weights
)

model = lgb.train(
    final_params,
    dtrain_final,
    num_boost_round=final_num_boost_round,
    callbacks=[
        lgb.log_evaluation(0)
    ]
)

# ================================================================
# 13. PREDICCIONES
# ================================================================

y_train_pred = model.predict(
    X_train_full
)

y_test_pred = model.predict(
    X_test
)

# ================================================================
# 14. BASELINE NAIVE
# ================================================================
#
# Baseline:
#
#     y_hat_t = y_(t-1)
#
# Como casos_lag_1 = y_(t-1), recuperamos directamente la variable.
# ================================================================

naive_test_pred = X_test["casos_lag_1"].values

naive_mae = mean_absolute_error(
    y_test,
    naive_test_pred
)

naive_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        naive_test_pred
    )
)

naive_r2 = r2_score(
    y_test,
    naive_test_pred
)

# ================================================================
# 15. MÉTRICAS DEL MODELO
# ================================================================

train_mae = mean_absolute_error(
    y_train_full,
    y_train_pred
)

test_mae = mean_absolute_error(
    y_test,
    y_test_pred
)

train_rmse = np.sqrt(
    mean_squared_error(
        y_train_full,
        y_train_pred
    )
)

test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_pred
    )
)

train_r2 = r2_score(
    y_train_full,
    y_train_pred
)

test_r2 = r2_score(
    y_test,
    y_test_pred
)

# ------------------------------------------------
# Peak MAE
#
# IMPORTANTE:
# El umbral de pico viene de TRAIN, no del TEST.
# ------------------------------------------------

peak_mask_train = (
    y_train_full.values >= peak_threshold
)

peak_mask_test = (
    y_test.values >= peak_threshold
)

peak_mae_train = (
    mean_absolute_error(
        y_train_full.values[peak_mask_train],
        y_train_pred[peak_mask_train]
    )
    if peak_mask_train.sum() > 0
    else np.nan
)

peak_mae_test = (
    mean_absolute_error(
        y_test.values[peak_mask_test],
        y_test_pred[peak_mask_test]
    )
    if peak_mask_test.sum() > 0
    else np.nan
)

# Sesgo medio:
# positivo = tendencia a sobreestimar
# negativo = tendencia a subestimar
bias_test = np.mean(
    y_test_pred - y_test.values
)

# MAE en picos del baseline
naive_peak_mae = (
    mean_absolute_error(
        y_test.values[peak_mask_test],
        naive_test_pred[peak_mask_test]
    )
    if peak_mask_test.sum() > 0
    else np.nan
)

# ================================================================
# 16. RESULTADOS
# ================================================================

print("\n" + "=" * 80)
print("RESULTADOS FINALES — MODELO SIN DATA LEAKAGE")
print("=" * 80)

print(f"MAE Train       : {train_mae:.4f}")
print(f"MAE Test        : {test_mae:.4f}")
print(f"Peak MAE Train  : {peak_mae_train:.4f}")
print(f"Peak MAE Test   : {peak_mae_test:.4f}")
print(f"RMSE Train      : {train_rmse:.4f}")
print(f"RMSE Test       : {test_rmse:.4f}")
print(f"R² Train        : {train_r2:.4f}")
print(f"R² Test         : {test_r2:.4f}")
print(f"Bias Test       : {bias_test:.4f}")

print("\nBASELINE NAIVE — y(t) = y(t-1)")
print(f"MAE Test        : {naive_mae:.4f}")
print(f"RMSE Test       : {naive_rmse:.4f}")
print(f"R² Test         : {naive_r2:.4f}")
print(f"Peak MAE Test   : {naive_peak_mae:.4f}")

if naive_mae > 0:
    improvement_vs_naive = (
        (naive_mae - test_mae) /
        naive_mae
    ) * 100
else:
    improvement_vs_naive = np.nan

print(
    f"\nMejora MAE frente a baseline: "
    f"{improvement_vs_naive:.2f}%"
)

print("=" * 80)

# ================================================================
# 17. IMPORTANCIA DE FEATURES DE LIGHTGBM
# ================================================================

lgb_importance = pd.DataFrame({
    "Feature": selected_features_rf,
    "Importance_gain": model.feature_importance(
        importance_type="gain"
    ),
    "Importance_split": model.feature_importance(
        importance_type="split"
    )
}).sort_values(
    "Importance_gain",
    ascending=False
)

# ================================================================
# 18. TABLA DE PREDICCIONES
# ================================================================

pred_df = pd.DataFrame({
    "fecha": fechas_test,
    "año": años_test,
    "semana_epi": semanas_test,
    "casos_reales": y_test.values,
    "predicciones_lightgbm": y_test_pred,
    "prediccion_naive": naive_test_pred,
})

pred_df["error_absoluto"] = (
    np.abs(
        pred_df["casos_reales"] -
        pred_df["predicciones_lightgbm"]
    )
)

pred_df["error"] = (
    pred_df["predicciones_lightgbm"] -
    pred_df["casos_reales"]
)

pred_df["es_pico"] = (
    pred_df["casos_reales"] >= peak_threshold
)

# ================================================================
# 19. ANÁLISIS POR RANGO
# ================================================================

max_case = max(
    200,
    int(np.ceil(y_test.max() / 50) * 50)
)

bins = [
    -np.inf,
    5,
    10,
    20,
    50,
    100,
    200,
    max_case
]

labels = [
    "0-5",
    "5-10",
    "10-20",
    "20-50",
    "50-100",
    "100-200",
    f">200"
]

# Eliminar duplicados de bins si el máximo es pequeño
bins_unique = []
labels_unique = []

for i, b in enumerate(bins):
    if len(bins_unique) == 0 or b > bins_unique[-1]:
        bins_unique.append(b)
        if i > 0:
            labels_unique.append(labels[i-1])

if len(labels_unique) != len(bins_unique) - 1:
    labels_unique = [
        str(bins_unique[i]) + "-" + str(bins_unique[i+1])
        for i in range(len(bins_unique)-1)
    ]

pred_df["rango_casos"] = pd.cut(
    pred_df["casos_reales"],
    bins=bins_unique,
    labels=labels_unique,
    include_lowest=True
)

error_analysis_rows = []

for label in pred_df["rango_casos"].dropna().unique():

    mask = pred_df["rango_casos"] == label

    if mask.sum() > 0:

        error_analysis_rows.append({
            "Rango": str(label),
            "Count": int(mask.sum()),
            "MAE": pred_df.loc[
                mask,
                "error_absoluto"
            ].mean(),
            "RMSE": np.sqrt(
                np.mean(
                    pred_df.loc[
                        mask,
                        "error"
                    ] ** 2
                )
            ),
            "Max_Error": pred_df.loc[
                mask,
                "error_absoluto"
            ].max()
        })

error_analysis = pd.DataFrame(
    error_analysis_rows
)

# ================================================================
# 20. GUARDAR DATASET PROCESADO
# ================================================================

processed_file = os.path.join(
    processed_dir,
    "dataset_procesado_sin_leakage.xlsx"
)

columns_to_save = (
    exclude_cols +
    [target_col] +
    selected_features_rf
)

columns_to_save = [
    col for col in columns_to_save
    if col in df_engineered.columns
]

df_final = df_engineered[
    columns_to_save
].copy()

df_final.to_excel(
    processed_file,
    index=False
)

# ================================================================
# 21. GUARDAR RESULTADOS EN EXCEL
# ================================================================

excel_file = os.path.join(
    output_dir,
    "resultados_modelo_sin_leakage.xlsx"
)

metrics_df = pd.DataFrame({
    "Métrica": [
        "MAE",
        "RMSE",
        "R²",
        "Peak MAE 80% TRAIN",
        "Bias"
    ],
    "LightGBM Train": [
        train_mae,
        train_rmse,
        train_r2,
        peak_mae_train,
        np.mean(
            y_train_pred -
            y_train_full.values
        )
    ],
    "LightGBM Test 2026": [
        test_mae,
        test_rmse,
        test_r2,
        peak_mae_test,
        bias_test
    ],
    "Baseline Naive Test 2026": [
        naive_mae,
        naive_rmse,
        naive_r2,
        naive_peak_mae,
        np.mean(
            naive_test_pred -
            y_test.values
        )
    ]
})

params_df = pd.DataFrame({
    "Parámetro": list(best_params.keys()),
    "Valor": [
        str(v)
        for v in best_params.values()
    ]
})

walk_forward_df = pd.DataFrame({
    "Fold": np.arange(
        1,
        len(folds) + 1
    ),
    "Train": [
        str(f[2])
        for f in folds
    ],
    "Validacion": [
        f[3]
        for f in folds
    ],
    "Best_iteration": best_iterations
})

with pd.ExcelWriter(
    excel_file,
    engine="openpyxl"
) as writer:

    metrics_df.to_excel(
        writer,
        sheet_name="Metricas",
        index=False
    )

    pred_df.to_excel(
        writer,
        sheet_name="Predicciones_Test",
        index=False
    )

    feature_importance.to_excel(
        writer,
        sheet_name="Features_RF",
        index=False
    )

    lgb_importance.to_excel(
        writer,
        sheet_name="Features_LightGBM",
        index=False
    )

    params_df.to_excel(
        writer,
        sheet_name="Parametros",
        index=False
    )

    walk_forward_df.to_excel(
        writer,
        sheet_name="Walk_Forward",
        index=False
    )

    error_analysis.to_excel(
        writer,
        sheet_name="Analisis_Errores",
        index=False
    )

# ================================================================
# 22. GUARDAR MODELO
# ================================================================

model_file = os.path.join(
    output_dir,
    "modelo_lightgbm_final_sin_leakage.txt"
)

model.save_model(
    model_file
)

# ================================================================
# 23. GRÁFICOS
# ================================================================

import matplotlib.pyplot as plt

fig, ax = plt.subplots(
    figsize=(16, 6)
)

ax.plot(
    fechas_test,
    y_test.values,
    label="Real",
    linewidth=1.5
)

ax.plot(
    fechas_test,
    y_test_pred,
    label="LightGBM",
    linewidth=1.5
)

ax.plot(
    fechas_test,
    naive_test_pred,
    label="Baseline naive",
    linewidth=1.2,
    linestyle="--"
)

ax.scatter(
    fechas_test[peak_mask_test],
    y_test.values[peak_mask_test],
    s=30,
    label="Picos definidos con TRAIN"
)

ax.set_xlabel("Fecha")
ax.set_ylabel("Casos de dengue")

ax.set_title(
    f"Predicción 2026 — "
    f"MAE LightGBM = {test_mae:.2f} | "
    f"MAE Naive = {naive_mae:.2f}"
)

ax.legend()
ax.grid(True, alpha=0.3)

plt.xticks(rotation=45)
plt.tight_layout()

plot_file = os.path.join(
    output_dir,
    "comparativa_test_2026_sin_leakage.png"
)

plt.savefig(
    plot_file,
    dpi=300,
    bbox_inches="tight"
)

plt.close()

# ================================================================
# 24. RESUMEN FINAL
# ================================================================

print("\n" + "=" * 80)
print("RESUMEN FINAL")
print("=" * 80)

print(
    f"✓ Features iniciales: "
    f"{len(predictor_cols)}"
)

print(
    f"✓ Features después de ingeniería: "
    f"{len(new_predictor_cols)}"
)

print(
    f"✓ Features seleccionadas por RF: "
    f"{len(selected_features_rf)}"
)

print(
    f"✓ MAE Train: "
    f"{train_mae:.2f}"
)

print(
    f"✓ MAE Test 2026: "
    f"{test_mae:.2f}"
)

print(
    f"✓ Peak MAE Test 2026: "
    f"{peak_mae_test:.2f}"
)

print(
    f"✓ R² Test 2026: "
    f"{test_r2:.4f}"
)

print(
    f"✓ MAE Baseline Naive: "
    f"{naive_mae:.2f}"
)

print(
    f"✓ Mejora frente al baseline: "
    f"{improvement_vs_naive:.2f}%"
)

print("\nArchivos generados:")

print(
    f"  - Dataset procesado: "
    f"{processed_file}"
)

print(
    f"  - Features RF: "
    f"{features_file}"
)

print(
    f"  - Resultados Excel: "
    f"{excel_file}"
)

print(
    f"  - Modelo LightGBM: "
    f"{model_file}"
)

print(
    f"  - Gráfico: "
    f"{plot_file}"
)

print("=" * 80)
print("PROCESO TERMINADO")
print("=" * 80)




CARGANDO DATOS
Registros originales: 270
Predictores meteorológicos/iniciales: 168
Periodo: 2021-03-28 00:00:00 → 2026-05-31 00:00:00

INGENIERÍA DE ATRIBUTOS SIN DATA LEAKAGE

SEPARACIÓN TEMPORAL
TRAIN: 249 registros
TEST : 21 registros
Umbral pico (80% TRAIN): 40.000
Umbral extremo (95% TRAIN): 74.000

DATOS DESPUÉS DE INGENIERÍA
Predictores: 235
TRAIN: 197
TEST : 21

SELECCIÓN DE FEATURES CON RANDOM FOREST — SOLO TRAIN


[I 2026-08-31 19:37:56,617] A new study created in memory with name: no-name-0d42bddb-672f-449e-bb48-a95c44a95804


Features iniciales: 235
Features seleccionadas: 30

Top 15:
 1. casos_dengue_lag_1                       0.045216
 2. roll_mean_3                              0.044778
 3. roll_max_7                               0.042446
 4. roll_min_3                               0.040424
 5. temp_max_x_casos_lag1                    0.040226
 6. casos_lag_1                              0.038932
 7. casos_dengue_lag_2                       0.034646
 8. roll_max_3                               0.034331
 9. temp_min_x_casos_lag1                    0.033744
10. roll_mean_5                              0.033686
11. temp_x_casos_lag1                        0.032781
12. casos_lag_2                              0.032551
13. roll_max_5                               0.027032
14. es_pico_lag1                             0.022188
15. roll_min_5                               0.021731

FOLDS WALK-FORWARD
Fold 1: Train [np.int64(2022), np.int64(2023)] → Validación 2024
Fold 2: Train [np.int64(2022), np.int64(2023)

Best trial: 0. Best value: 17.8022:   4%|▍         | 2/50 [00:00<00:07,  6.30it/s]

[I 2026-08-31 19:37:56,793] Trial 0 finished with value: 17.802155067590192 and parameters: {'num_leaves': 36, 'learning_rate': 0.044635901521768134, 'feature_fraction': 0.892797576724562, 'bagging_fraction': 0.8394633936788146, 'bagging_freq': 2, 'min_child_samples': 10, 'reg_alpha': 0.11616722433639892, 'reg_lambda': 1.7323522915498704, 'min_split_gain': 0.3005575058716044, 'max_depth': 10, 'subsample': 0.608233797718321, 'colsample_bytree': 0.9879639408647978}. Best is trial 0 with value: 17.802155067590192.
[I 2026-08-31 19:37:56,938] Trial 1 finished with value: 21.117448004960647 and parameters: {'num_leaves': 69, 'learning_rate': 0.008152843673110739, 'feature_fraction': 0.6727299868828402, 'bagging_fraction': 0.6733618039413735, 'bagging_freq': 4, 'min_child_samples': 23, 'reg_alpha': 0.8638900372842315, 'reg_lambda': 0.5824582803960838, 'min_split_gain': 0.30592644736118974, 'max_depth': 4, 'subsample': 0.7168578594140873, 'colsample_bytree': 0.7465447373174767}. Best is trial

...

[I 2026-08-31 19:38:08,274] Trial 47 finished with value: 18.128177941922477 and parameters: {'num_leaves': 29, 'learning_rate': 0.007228594044962958, 'feature_fraction': 0.9471835526857915, 'bagging_fraction': 0.9483400083914335, 'bagging_freq': 5, 'min_child_samples': 11, 'reg_alpha': 1.8139054934232162, 'reg_lambda': 0.6182719874646805, 'min_split_gain': 0.20962202596961937, 'max_depth': 11, 'subsample': 0.9439047094735391, 'colsample_bytree': 0.8829353510624081}. Best is trial 21 with value: 17.401302648860444.


Best trial: 21. Best value: 17.4013:  98%|█████████▊| 49/50 [00:12<00:00,  3.33it/s]

[I 2026-08-31 19:38:08,616] Trial 48 finished with value: 17.63714588984624 and parameters: {'num_leaves': 10, 'learning_rate': 0.03466609994712628, 'feature_fraction': 0.9167360881813689, 'bagging_fraction': 0.8627463531050197, 'bagging_freq': 8, 'min_child_samples': 8, 'reg_alpha': 0.96374572973511, 'reg_lambda': 0.09453429191004181, 'min_split_gain': 0.4226606275574531, 'max_depth': 8, 'subsample': 0.7242572707260966, 'colsample_bytree': 0.7049489094638829}. Best is trial 21 with value: 17.401302648860444.


Best trial: 21. Best value: 17.4013: 100%|██████████| 50/50 [00:12<00:00,  4.01it/s]


[I 2026-08-31 19:38:09,071] Trial 49 finished with value: 19.57438522404675 and parameters: {'num_leaves': 71, 'learning_rate': 0.00513790183937535, 'feature_fraction': 0.9226378214906894, 'bagging_fraction': 0.8157381112798678, 'bagging_freq': 8, 'min_child_samples': 17, 'reg_alpha': 0.9075959167081442, 'reg_lambda': 0.11873040253125966, 'min_split_gain': 0.43258988030522527, 'max_depth': 8, 'subsample': 0.7741947618776647, 'colsample_bytree': 0.6320912522260612}. Best is trial 21 with value: 17.401302648860444.

Mejores parámetros:
  num_leaves: 14
  learning_rate: 0.03643119977148973
  feature_fraction: 0.8608775194487166
  bagging_fraction: 0.8840284403801686
  bagging_freq: 8
  min_child_samples: 8
  reg_alpha: 0.952050793779362
  reg_lambda: 0.14569137860993836
  min_split_gain: 0.49923595701136286
  max_depth: 9
  subsample: 0.7429296463367473
  colsample_bytree: 0.6811074498539247

Mejor score walk-forward: 17.4013

ENTRENAMIENTO FINAL
Mejores iteraciones por fold: [304, 189]
N